In [1]:
import importlib
import json
import pandas as pd
from pathlib import Path
import humaidclf.report as report

importlib.reload(report)

# Collect all results from the curated results/ tree
entries = report._collect_results(
    Path("results"),
    recompute=False,
    recompute_missing_only=True,
    recompute_subdir="analysis_recomputed",
)
df = pd.DataFrame(entries)

# Filter: test split, gpt-4o / gpt-4o-mini, RULES1 only
df = df[
    (df["split"] == "test")
    & (df["model"].isin(["gpt-4o", "gpt-4o-mini"]))
    & (df["run_name"].str.contains("RULES1", case=False))
].copy()

# Keep best F1 per (model, event)
df = (
    df.sort_values("macro_f1", ascending=False)
    .drop_duplicates(subset=["model", "event"], keep="first")
    .sort_values(["model", "event"])
    .reset_index(drop=True)
)

df[["event", "model", "run_name", "test_size", "accuracy", "macro_f1"]]

,event,model,run_name,test_size,accuracy,macro_f1
0,california_wildfires_2018,gpt-4o,20251104-042558-modeS-gpt-4o-RULES1-filtered-s...,1461,0.712526,0.628638
1,canada_wildfires_2016,gpt-4o,20251104-010418-modeS-gpt-4o-RULES1-filtered-s...,445,0.786517,0.672589
2,cyclone_idai_2019,gpt-4o,20251104-013437-modeS-gpt-4o-RULES1-filtered-s...,779,0.753530,0.640507
3,hurricane_dorian_2019,gpt-4o,20251104-054636-modeS-gpt-4o-RULES1-filtered-s...,1508,0.637268,0.598173
4,hurricane_florence_2018,gpt-4o,20251104-023000-modeS-gpt-4o-RULES1-filtered-s...,1241,0.768735,0.707257
5,hurricane_harvey_2017,gpt-4o,20251104-121414-modeS-gpt-4o-RULES1-filtered-s...,1805,0.657618,0.612985
6,hurricane_irma_2017,gpt-4o,20251104-073738-modeS-gpt-4o-RULES1-filtered-s...,1862,0.634801,0.617246
7,hurricane_maria_2017,gpt-4o,20251104-031023-modeS-gpt-4o-RULES1-filtered-s...,1442,0.667129,0.634095
8,kaikoura_earthquake_2016,gpt-4o,20251113-163546-modeS-gpt-4o-RULES1-filtered-s...,435,0.733333,0.745308
9,kerala_floods_2018,gpt-4o,20251104-093125-modeS-gpt-4o-RULES1-filtered-s...,1582,0.690898,0.559923


In [2]:
RESULTS_ROOT = Path("results")
IMG_FILES = report.IMG_FILES


def rel(p):
    """Return a path relative to results/ with POSIX separators."""
    return str(Path(p).relative_to(RESULTS_ROOT)).replace("\\", "/")


def render_model_table(df_model, model_name):
    """Render a summary table for one model, with an Average row at the bottom."""
    tbl = df_model.sort_values("event").reset_index(drop=True)

    # Best-run flags
    max_acc = tbl["accuracy"].max()
    max_f1 = tbl["macro_f1"].max()

    # Header
    headers = [
        ("Event", "string"),
        ("Run", "string"),
        ("Test size", "number"),
        ("Accuracy", "number"),
        ("Macro-F1", "number"),
    ]
    head_cells = "".join(
        f"<th data-type='{t}'>{h}</th>" for h, t in headers
    )

    # Data rows with group striping by event
    rows = []
    grp = 0
    prev_event = None
    for _, r in tbl.iterrows():
        if r["event"] != prev_event:
            grp ^= 1
            prev_event = r["event"]

        best_acc_badge = (
            "<span class='badge badge-acc' title='Best Accuracy'>best</span>"
            if r["accuracy"] >= max_acc - 1e-12
            else ""
        )
        best_f1_badge = (
            "<span class='badge badge-f1' title='Best Macro-F1'>best</span>"
            if r["macro_f1"] >= max_f1 - 1e-12
            else ""
        )

        rows.append(
            f"<tr class='grp-{grp}'>"
            f"<td><strong>{r['event']}</strong></td>"
            f"<td><code>{r['run_name']}</code></td>"
            f"<td class='num' data-sort='{int(r['test_size'])}'>{int(r['test_size'])}</td>"
            f"<td class='num' data-sort='{r['accuracy']:.6f}'>{r['accuracy']:.4f} {best_acc_badge}</td>"
            f"<td class='num' data-sort='{r['macro_f1']:.6f}'>{r['macro_f1']:.4f} {best_f1_badge}</td>"
            "</tr>"
        )

    # Average row
    avg_acc = tbl["accuracy"].mean()
    avg_f1 = tbl["macro_f1"].mean()
    total_size = int(tbl["test_size"].sum())
    rows.append(
        "<tr class='avg-row'>"
        "<td><strong>Average</strong></td>"
        "<td></td>"
        f"<td class='num' data-sort='{total_size}'>{total_size}</td>"
        f"<td class='num' data-sort='{avg_acc:.6f}'><strong>{avg_acc:.4f}</strong></td>"
        f"<td class='num' data-sort='{avg_f1:.6f}'><strong>{avg_f1:.4f}</strong></td>"
        "</tr>"
    )

    return (
        f"<table class='summary sortable'>"
        f"<thead><tr>{head_cells}</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody>"
        f"</table>"
    )


def render_detail_cards(df_model, model_name):
    """Render detail cards with embedded chart images for one model."""
    cards = []
    tbl = df_model.sort_values("event").reset_index(drop=True)

    max_acc = tbl["accuracy"].max()
    max_f1 = tbl["macro_f1"].max()

    for _, r in tbl.iterrows():
        charts = Path(r["charts_dir"])
        imgs = []
        for name in IMG_FILES:
            fp = charts / name
            if fp.exists():
                imgs.append(
                    f'<div class="imgbox">'
                    f'  <img class="zoomable" src="{rel(fp)}" alt="{name}" '
                    f'       data-fullsrc="{rel(fp)}">'
                    f'</div>'
                )
        imgs_html = "\n".join(imgs) if imgs else "<em>No charts found.</em>"

        best_acc_badge = (
            "<span class='badge badge-acc' title='Best Accuracy'>best</span>"
            if r["accuracy"] >= max_acc - 1e-12
            else ""
        )
        best_f1_badge = (
            "<span class='badge badge-f1' title='Best Macro-F1'>best</span>"
            if r["macro_f1"] >= max_f1 - 1e-12
            else ""
        )

        cards.append(f"""
        <section class="card">
          <div class="head">
            <div>
              <h3>{r['event']} &mdash; test</h3>
              <div class="sub">model: <code>{model_name}</code> &mdash; <code>{r['run_name']}</code></div>
              <div class="sub path">{rel(r['dir'])}</div>
            </div>
            <table class="metrics">
              <tr><th>Test size</th><td>{int(r['test_size'])}</td></tr>
              <tr><th>Accuracy</th><td>{r['accuracy']:.4f} {best_acc_badge}</td></tr>
              <tr><th>Macro-F1</th><td>{r['macro_f1']:.4f} {best_f1_badge}</td></tr>
            </table>
          </div>
          <div class="imgs">
            {imgs_html}
          </div>
        </section>
        """)

    return "\n".join(cards)


# Build sections for each model
models = ["gpt-4o", "gpt-4o-mini"]
summary_sections = []
detail_sections = []

for model in models:
    df_m = df[df["model"] == model]
    avg_acc = df_m["accuracy"].mean()
    avg_f1 = df_m["macro_f1"].mean()

    summary_sections.append(f"""
    <section class="split-summary">
      <h2>Model: <code>{model}</code>
        <span class="avg-inline">Avg Accuracy: <strong>{avg_acc:.4f}</strong> &nbsp;|&nbsp; Avg Macro-F1: <strong>{avg_f1:.4f}</strong></span>
      </h2>
      {render_model_table(df_m, model)}
    </section>
    """)

    detail_sections.append(f"""
    <section class="split-detail">
      <h2>Model: <code>{model}</code> &mdash; Detail Cards</h2>
      <div class="grid">
        {render_detail_cards(df_m, model)}
      </div>
    </section>
    """)

summary_block = "\n".join(summary_sections)
details_block = "\n".join(detail_sections)

# Assemble full HTML page
html = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>HumAID Test Split &mdash; RULES1 Dashboard</title>
<style>
  body {{ font-family: system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial, sans-serif; margin: 24px; }}
  h1 {{ margin-top: 0; }}
  h2 {{ margin: 18px 0 8px; }}
  .avg-inline {{ font-size: 14px; font-weight: 400; color: #374151; margin-left: 16px; }}
  .summary {{ width: 100%; border-collapse: collapse; margin-bottom: 18px; }}
  .summary th, .summary td {{ border: 1px solid #e5e7eb; padding: 8px 10px; }}
  .summary th {{ background: #f9fafb; text-align: left; cursor: pointer; }}
  .summary td.num {{ text-align: right; font-variant-numeric: tabular-nums; }}
  .grid {{ display: grid; grid-template-columns: 1fr; gap: 18px; }}
  .card {{ border: 1px solid #e5e7eb; border-radius: 12px; padding: 16px; background: #fff; box-shadow: 0 1px 2px rgba(0,0,0,0.03); }}
  .head {{ display: flex; justify-content: space-between; align-items: flex-start; gap: 16px; flex-wrap: wrap; }}
  .sub {{ color: #6b7280; font-size: 12px; }}
  .sub.path {{ font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; }}
  table.metrics {{ border-collapse: collapse; }}
  table.metrics th {{ text-align: left; padding-right: 8px; color: #374151; }}
  table.metrics td {{ text-align: right; font-weight: 600; color: #111827; }}
  .imgs {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(280px, 1fr)); gap: 12px; margin-top: 12px; }}
  .imgbox {{ border: 1px solid #eee; border-radius: 8px; padding: 8px; background: #fafafa; }}
  .imgbox img {{ width: 100%; height: auto; display: block; cursor: zoom-in; }}

  /* Badges */
  .badge {{ display: inline-block; padding: 2px 6px; border-radius: 999px; font-size: 10px; margin-left: 6px; vertical-align: 1px; }}
  .badge-acc {{ background: #6efffa; color: #0c484a; border: 1px solid #0c484a; }}
  .badge-f1  {{ background: #92fcaf; color: #0e3318; border: 1px solid #0e3318; }}

  /* Group striping by event */
  tr.grp-0 {{ background: #fff7ed; }}
  tr.grp-1 {{ background: #eaf2ff; }}
  tr.grp-0:hover, tr.grp-1:hover {{ background: #e6f25e; }}
  tr.grp-0 td:first-child {{ border-left: 4px solid #60a5fa; }}
  tr.grp-1 td:first-child {{ border-left: 4px solid #34d399; }}

  /* Average row */
  tr.avg-row {{ background: #f3f4f6; border-top: 2px solid #9ca3af; }}
  tr.avg-row td {{ font-weight: 700; }}
  tr.avg-row:hover {{ background: #e5e7eb; }}

  /* Modal (for chart zoom) */
  .modal {{
    position: fixed; inset: 0; display: none;
    background: rgba(0,0,0,0.7); z-index: 9999;
    align-items: center; justify-content: center;
    padding: 24px;
  }}
  .modal.open {{ display: flex; }}
  .modal img {{
    max-width: 95vw; max-height: 85vh;
    box-shadow: 0 10px 30px rgba(0,0,0,0.4);
    border-radius: 8px; background: #fff;
  }}
  .modal .close {{
    position: absolute; top: 12px; right: 16px;
    font-size: 28px; color: #fff; cursor: pointer; user-select: none;
  }}
</style>
</head>
<body>
  <h1>HumAID Zero-shot Results &mdash; Test Split, RULES1</h1>

  <section>
    <h2>Summary by Model</h2>
    {summary_block}
  </section>

  <hr style="margin:24px 0; border:none; border-top:1px solid #e5e7eb;" />

  <section>
    <h2>Details by Model</h2>
    {details_block}
  </section>

  <!-- Shared Modal -->
  <div id="imgModal" class="modal" aria-hidden="true">
    <span class="close" title="Close (Esc)">&times;</span>
    <div id="modalPanel"></div>
  </div>

  <script>
  (function() {{
    const modal = document.getElementById('imgModal');
    const panel = document.getElementById('modalPanel');
    const closeBtn = modal.querySelector('.close');

    function openModal(src) {{
      panel.innerHTML = '<img alt="Preview" src="' + src + '">';
      modal.classList.add('open');
      modal.setAttribute('aria-hidden', 'false');
    }}

    function closeModal() {{
      modal.classList.remove('open');
      modal.setAttribute('aria-hidden', 'true');
      panel.innerHTML = '';
    }}

    document.addEventListener('click', function(e) {{
      const img = e.target.closest('img.zoomable');
      if (img) {{
        openModal(img.getAttribute('data-fullsrc') || img.src);
        return;
      }}
      if (e.target === modal || e.target === closeBtn) {{
        closeModal();
      }}
    }});

    document.addEventListener('keydown', function(e) {{
      if (e.key === 'Escape' && modal.classList.contains('open')) {{
        closeModal();
      }}
    }});
  }})();

  // Sortable tables
  (function() {{
    function compare(a, b, type) {{
      if (type === 'number') {{
        const x = parseFloat(a); const y = parseFloat(b);
        if (isNaN(x) && isNaN(y)) return 0;
        if (isNaN(x)) return -1;
        if (isNaN(y)) return 1;
        return x - y;
      }}
      return ('' + a).localeCompare(('' + b), undefined, {{ sensitivity: 'base' }});
    }}

    document.querySelectorAll('table.sortable').forEach(function(table) {{
      const thead = table.querySelector('thead');
      if (!thead) return;
      const directions = [];
      thead.addEventListener('click', function(e) {{
        const th = e.target.closest('th');
        if (!th) return;
        const idx = Array.prototype.indexOf.call(th.parentNode.children, th);
        const dtype = th.getAttribute('data-type') || 'string';
        directions[idx] = directions[idx] ? -directions[idx] : 1;
        const tbody = table.querySelector('tbody');
        const rows = Array.from(tbody.querySelectorAll('tr:not(.avg-row)'));
        const avgRow = tbody.querySelector('tr.avg-row');
        rows.sort(function(r1, r2) {{
          const c1 = r1.children[idx];
          const c2 = r2.children[idx];
          const v1 = c1.getAttribute('data-sort') || c1.textContent.trim();
          const v2 = c2.getAttribute('data-sort') || c2.textContent.trim();
          return directions[idx] * compare(v1, v2, dtype);
        }});
        rows.forEach(function(r) {{ tbody.appendChild(r); }});
        if (avgRow) tbody.appendChild(avgRow);
      }});
    }});
  }})();
  </script>

  <footer style="margin-top:24px; color:#6b7280; font-size:12px;">
    Generated automatically. Click any chart to zoom. Press ESC to close. Click headers to sort tables.
  </footer>
</body>
</html>
"""

out_path = RESULTS_ROOT / "test_rules1_dashboard.html"
out_path.write_text(html, encoding="utf-8")
print(f"Dashboard written to: {out_path}")
print(f"Total entries: {len(df)} ({', '.join(f'{m}: {len(df[df.model==m])}' for m in models)})")

Dashboard written to: results\test_rules1_dashboard.html
Total entries: 20 (gpt-4o: 10, gpt-4o-mini: 10)


In [3]:
from IPython.display import IFrame

IFrame(src="results/test_rules1_dashboard.html", width="100%", height=900)